# 📊 TrendSense — Notebook 4: Results Visualization

**Objective:** Generate publication-quality figures for the project report. Summarize findings, demonstrate decision engine, and create final visualizations.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

import config
from src.data_ingestion import download_walmart_data, fetch_google_trends
from src.tvi import compute_tvi_features, get_tvi_summary
from src.feature_engineering import merge_all_features, get_feature_columns
from src.models import temporal_train_test_split, train_xgboost, train_random_forest, train_arima_baseline
from src.decision_engine import generate_decision, generate_batch_decisions, get_decision_summary, format_decision_output
from src.lag_optimizer import optimize_lags_by_category

sns.set_theme(style='whitegrid', font_scale=1.2)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 150
print('✅ Libraries loaded')

## 1. Load Data & Train Models

In [ ]:
# Load data
walmart_df = download_walmart_data()
trends_df = fetch_google_trends()

# Feature engineering
featured_df = merge_all_features(walmart_df, trends_df, category='General')
feature_cols = get_feature_columns(featured_df, 'Weekly_Sales')

# Train-test split
X_train, X_test, y_train, y_test = temporal_train_test_split(
    featured_df, 'Weekly_Sales', feature_cols
)

# Train models
train_series = featured_df.sort_values('Date')['Weekly_Sales'].iloc[:len(X_train)]
test_series = featured_df.sort_values('Date')['Weekly_Sales'].iloc[len(X_train):]

arima_result = train_arima_baseline(train_series, test_series)
rf_result = train_random_forest(X_train, y_train, X_test, y_test)
xgb_result = train_xgboost(X_train, y_train, X_test, y_test)

print('\n✅ All models trained')

## 2. Figure 1: Model Comparison (Publication Quality)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models_names = ['ARIMA\n(Baseline)', 'Random\nForest', 'XGBoost\n+ TVI']
colors = ['#EF4444', '#F59E0B', '#10B981']

metrics_data = {
    'MAPE (%)': [arima_result['metrics']['MAPE (%)'], rf_result['metrics']['MAPE (%)'], xgb_result['metrics']['MAPE (%)']],
    'RMSE': [arima_result['metrics']['RMSE'], rf_result['metrics']['RMSE'], xgb_result['metrics']['RMSE']],
    'R²': [arima_result['metrics']['R²'], rf_result['metrics']['R²'], xgb_result['metrics']['R²']],
}

for idx, (metric, values) in enumerate(metrics_data.items()):
    bars = axes[idx].bar(models_names, values, color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
    axes[idx].set_title(metric, fontweight='bold', fontsize=14)
    for bar, val in zip(bars, values):
        axes[idx].text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(values)*0.02,
                       f'{val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    axes[idx].set_ylim(0, max(values) * 1.2)

plt.suptitle('TrendSense — Model Performance Comparison', fontweight='bold', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, 'fig1_model_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()
print('💾 Saved: fig1_model_comparison.png')

## 3. Figure 2: Decision Engine Demo

In [ ]:
# Demo scenarios
scenarios = [
    {'name': 'Normal Week', 'demand': 10500, 'stock': 10000, 'spike': False, 'severity': 'NONE'},
    {'name': 'Pre-Diwali', 'demand': 13000, 'stock': 10000, 'spike': True, 'severity': 'MILD'},
    {'name': 'Diwali Week', 'demand': 16000, 'stock': 10000, 'spike': True, 'severity': 'MODERATE'},
    {'name': 'Big Billion Day', 'demand': 22000, 'stock': 10000, 'spike': True, 'severity': 'SEVERE'},
    {'name': 'Post-Season', 'demand': 9500, 'stock': 10000, 'spike': False, 'severity': 'NONE'},
]

print('🎯 Decision Engine Demo')
print('=' * 60)

for s in scenarios:
    decision = generate_decision(
        predicted_demand=s['demand'],
        current_stock=s['stock'],
        tvi_status={'is_spike': s['spike'], 'severity': s['severity'], 'tvi_value': 0},
        model_confidence=0.85,
        category='Festival'
    )
    icon = {'HOLD': '🟢', 'INCREASE STOCK': '🟡', 'URGENT RESTOCK': '🔴'}[decision['action']]
    print(f"\n{s['name']:20s} → {icon} {decision['action']:20s} (Confidence: {decision['confidence_pct']})")

In [ ]:
# Batch decision visualization
np.random.seed(42)
n = 50
predictions = np.random.uniform(8000, 20000, n)
stocks = np.full(n, 10000)
tvi_statuses = [{'is_spike': np.random.random() < 0.2,
                 'severity': np.random.choice(['MILD', 'MODERATE', 'SEVERE']) if np.random.random() < 0.2 else 'NONE',
                 'tvi_value': 0} for _ in range(n)]

batch_df = generate_batch_decisions(predictions, stocks, tvi_statuses)
summary = get_decision_summary(batch_df)

# Donut chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Donut
labels = ['HOLD', 'INCREASE\nSTOCK', 'URGENT\nRESTOCK']
sizes = [summary['hold_count'], summary['increase_count'], summary['urgent_count']]
colors_pie = ['#10B981', '#F59E0B', '#EF4444']
explode = (0, 0.05, 0.1)

wedges, texts, autotexts = axes[0].pie(sizes, explode=explode, labels=labels, colors=colors_pie,
                                        autopct='%1.1f%%', startangle=90, pctdistance=0.75,
                                        textprops={'fontweight': 'bold'})
centre_circle = plt.Circle((0,0), 0.5, fc='white')
axes[0].add_artist(centre_circle)
axes[0].set_title('Decision Distribution', fontweight='bold', fontsize=14)

# Confidence by decision type
for action, color in zip(['HOLD', 'INCREASE STOCK', 'URGENT RESTOCK'], colors_pie):
    mask = batch_df['action'] == action
    if mask.any():
        axes[1].hist(batch_df.loc[mask, 'confidence'] * 100, bins=10, alpha=0.6,
                     color=color, label=action, edgecolor='white')

axes[1].set_title('Confidence Score Distribution', fontweight='bold', fontsize=14)
axes[1].set_xlabel('Confidence (%)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, 'fig2_decision_engine.png'), dpi=300, bbox_inches='tight')
plt.show()
print('💾 Saved: fig2_decision_engine.png')

## 4. Figure 3: Category Lag Optimization

In [ ]:
# Category lag visualization
categories = list(config.DEFAULT_CATEGORY_LAGS.keys())
lags = list(config.DEFAULT_CATEGORY_LAGS.values())
rationale = ['Impulse buying', 'Research-driven', 'Moderate decision', 'Average behavior', 'Immediate need']

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['#EC4899', '#6366F1', '#F59E0B', '#3B82F6', '#10B981']
bars = ax.barh(categories, lags, color=colors_bar, alpha=0.85, edgecolor='white', linewidth=1.5, height=0.6)

for bar, lag, rat in zip(bars, lags, rationale):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{lag}d — {rat}', va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('Optimal Lag (days)', fontweight='bold')
ax.set_title('Category-Specific Lag Optimization\n(Social Signal → Purchase Delay)', fontweight='bold', fontsize=14)
ax.set_xlim(0, max(lags) + 10)

plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, 'fig3_lag_optimization.png'), dpi=300, bbox_inches='tight')
plt.show()
print('💾 Saved: fig3_lag_optimization.png')

## 5. Figure 4: TVI Innovation Explained

In [ ]:
# Compare raw volume vs TVI
np.random.seed(42)
weeks = 20
x = np.arange(weeks)

# Scenario: Two patterns with same endpoint but different velocity
gradual = np.linspace(30, 80, weeks) + np.random.normal(0, 2, weeks)
viral = np.concatenate([np.full(14, 30) + np.random.normal(0, 2, 14),
                        np.array([35, 50, 70, 85, 90, 80])])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Raw Volume
axes[0, 0].plot(x, gradual, 'b-o', linewidth=2, markersize=5, label='Gradual Growth', alpha=0.8)
axes[0, 0].plot(x, viral, 'r-s', linewidth=2, markersize=5, label='Viral Spike', alpha=0.8)
axes[0, 0].set_title('Raw Search Volume', fontweight='bold')
axes[0, 0].set_ylabel('Interest (0-100)')
axes[0, 0].legend()
axes[0, 0].annotate('Same endpoint!', xy=(19, 80), fontweight='bold', color='purple', fontsize=11)

# TVI
from src.tvi import compute_tvi
tvi_gradual = compute_tvi(pd.Series(gradual))
tvi_viral = compute_tvi(pd.Series(viral))

axes[0, 1].plot(x, tvi_gradual, 'b-o', linewidth=2, markersize=5, label='Gradual TVI', alpha=0.8)
axes[0, 1].plot(x, tvi_viral, 'r-s', linewidth=2, markersize=5, label='Viral TVI', alpha=0.8)
axes[0, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[0, 1].set_title('Trend Velocity Index (TVI)', fontweight='bold')
axes[0, 1].set_ylabel('TVI (%)')
axes[0, 1].legend()
axes[0, 1].annotate('VIRAL SPIKE\ndetected by TVI!', xy=(15, tvi_viral.iloc[15] if not pd.isna(tvi_viral.iloc[15]) else 50),
                     fontweight='bold', color='red', fontsize=11,
                     arrowprops=dict(arrowstyle='->', color='red'))

# Decision comparison
axes[1, 0].text(0.5, 0.7, 'Raw Volume Analysis', transform=axes[1, 0].transAxes,
                fontsize=16, ha='center', fontweight='bold')
axes[1, 0].text(0.5, 0.45, 'Both patterns look similar\nat the endpoint = 80', transform=axes[1, 0].transAxes,
                fontsize=12, ha='center', style='italic')
axes[1, 0].text(0.5, 0.2, '→ Same decision: HOLD', transform=axes[1, 0].transAxes,
                fontsize=14, ha='center', fontweight='bold', color='#F59E0B')
axes[1, 0].axis('off')

axes[1, 1].text(0.5, 0.7, 'TVI Analysis', transform=axes[1, 1].transAxes,
                fontsize=16, ha='center', fontweight='bold')
axes[1, 1].text(0.5, 0.45, 'Viral pattern has TVI spike\n= acceleration detected!', transform=axes[1, 1].transAxes,
                fontsize=12, ha='center', style='italic')
axes[1, 1].text(0.5, 0.2, '→ Correct: URGENT RESTOCK 🔴', transform=axes[1, 1].transAxes,
                fontsize=14, ha='center', fontweight='bold', color='#EF4444')
axes[1, 1].axis('off')

plt.suptitle('Why TVI Matters: Raw Volume vs Velocity', fontweight='bold', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, 'fig4_tvi_innovation.png'), dpi=300, bbox_inches='tight')
plt.show()
print('💾 Saved: fig4_tvi_innovation.png')

## 6. Summary Table for Report

In [ ]:
print('=' * 70)
print('TRENDSENSE — FINAL RESULTS SUMMARY')
print('=' * 70)
print(f'\n📊 Dataset: Walmart Store Sales ({len(walmart_df)} rows, {walmart_df["Store"].nunique()} stores)')
print(f'📈 Trends: Google Trends India ({len(trends_df)} weeks, {len([c for c in trends_df.columns if c != "date"])} keywords)')
print(f'🔧 Features: {len(feature_cols)} engineered features')
print(f'\n📋 Model Results:')
print(f'   ARIMA Baseline:  MAPE = {arima_result["metrics"]["MAPE (%)"]:.2f}%')
print(f'   Random Forest:   MAPE = {rf_result["metrics"]["MAPE (%)"]:.2f}%')
print(f'   XGBoost + TVI:   MAPE = {xgb_result["metrics"]["MAPE (%)"]:.2f}%')
print(f'\n   Δ-MAPE (XGBoost vs ARIMA): {arima_result["metrics"]["MAPE (%)"] - xgb_result["metrics"]["MAPE (%)"]:.2f}%')
print(f'\n🎯 Decision Engine: HOLD / INCREASE STOCK / URGENT RESTOCK with confidence scores')
print(f'⚡ Innovation: Trend Velocity Index (TVI) — measures acceleration, not volume')
print(f'⏱️ Category Lags: Fashion(3d) | Electronics(14d) | Festival(7d)')
print('\n' + '=' * 70)

---

## 📁 Generated Figures

All publication-quality figures saved to `reports/figures/`:

1. `fig1_model_comparison.png` — MAPE, RMSE, R² bar charts
2. `fig2_decision_engine.png` — Decision distribution and confidence
3. `fig3_lag_optimization.png` — Category-specific lag periods
4. `fig4_tvi_innovation.png` — Why TVI matters (Raw vs Velocity)

---
*TrendSense — AI Foundations for Engineers (CI124TA) — RVCE 2025*